# Ungraded Lab: Your First Model Deployment

## Task 1: Initial Configuration
Set up your environment for the segmentation model deployment. This model will help InsightlySoft's marketing team understand different customer groups based on their product usage patterns.

<b>Step 1:</b> Environment Setup



In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import pickle
import boto3
import os
import io


# # Load insightlysoft_dataset.csv and create a configuration that includes:

# # - Essential features for clustering

# # Load insightlysoft dataset
# df = # Your code here

# features =  # Your code here

Review your loaded data:
- Have you included relevant engagement metrics?

## Task 2: Data Preparation, Model Training and Saving

<b>Step 1:</b> Before training the model, ensure you are scaling the data.

In [ ]:
# Scale features
scaler = StandardScaler()
# Your code here

# Train clustering model with 3 segments
kmeans = # Your code here

# Fit the model
# Your code here

<b>Step 2:</b> Save your model into a S3 bucket.

In [ ]:
# Save the trained model and scaler locally in a pickle file
# Your code here

# Upload the saved model to S3
s3 = boto3.client('s3')
bucket_name =  # Replace with your own unique bucket name
s3_key = # Replace with your own key

s3.upload_file(model_path, bucket_name, s3_key)
print(f"Model uploaded to s3://{bucket_name}/{s3_key}")

## Task 3: Batch Processing Implementation
Implement the batch processing logic for customer segmentation.

<b>Step 1:</b>  Load model from S3

In [ ]:
# Load model directly from S3 into memory
# Your code here

# Unpack the model and scaler
kmeans, scaler = pickle.load(io.BytesIO(model_data))

<b>Step 2:</b> Implement Processing Logic

In [ ]:
# Load a sample of your dataset
# Remember to use features that have already been scaled for model inference
df_sample = df[]

# Start the batch processing
predictions = # Your code here

# Store your predictions in the df_sample as a new column
# Your code here

## Task 4: Results Validation and Reporting
Create tools to validate your segmentation results and generate useful reports for the marketing team.

In [ ]:
# Create validation and reporting functions that:
# - Verify segment assignments
# - Generate segment summaries
# - Identify key characteristics per segment
def analyze_segmentation_results(df):
    """
    Analyzes segmentation results and generates reports
    Returns: Analysis summary
    """
    # Your analysis code here
    pass

## Solution Code
Need a hand or are curious to compare your approach? Below is a complete solution you can use as a reference. This is just one of many valid ways to solve the problem. Make sure to give it a try on your own first. Use this implementation to troubleshoot, learn new techniques, or confirm your logic. Keep experimenting and enjoy the process!

## Task 1: Initial Configuration

<b>Step 1:</b> Environment Setup

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import pickle
import boto3
import os
import io

# Load insightlysoft_dataset.csv and create a configuration that includes:
# - Essential features for clustering

# Load insightlysoft dataset
df = pd.read_csv("insightlysoft_dataset.csv")

features =  ['monthly_login_freq', 'num_support_tickets',
             'avg_product_usage_hours', 'feature_adoption_score']
df[features].head()

## Task 2: Data Preparation, Model Training and Savi

<b>Step 1:</b> Before training the model, ensure you are scaling the data.

In [ ]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

# Train clustering model with 3 segments
kmeans = KMeans(n_clusters=3, random_state=42)

# Fit the model
kmeans.fit(X_scaled)

<b>Step 2:</b> Save your model into a S3 bucket.

In [ ]:
# Save the trained model and scaler locally in a pickle file
model_path = "segmentation_model.pkl"
with open(model_path, "wb") as f:
    pickle.dump((kmeans, scaler), f)

# Upload the saved model to S3
s3 = boto3.client('s3')
bucket_name = "insighlysoft-model" # Replace with your own unique bucket name
s3_key = "models/segmentation_model.pkl" # Replace with your own key

s3.upload_file(model_path, bucket_name, s3_key)
print(f"Model uploaded to s3://{bucket_name}/{s3_key}")

## Task 3: Batch Processing Implementation

<b>Step 1:</b> Load model from S3.

In [ ]:
# Load model directly from S3 into memory
response = s3.get_object(Bucket=bucket_name, Key=s3_key)
model_data = response['Body'].read()

# Unpack the model and scaler
kmeans_loaded, scaler_loaded = pickle.load(io.BytesIO(model_data))

<b>Step 2:</b> Implement Processing Logic

In [ ]:
# Load a sample of your dataset
# Remember to use features that have already been scaled for model inference
df_sample = df[features].sample(n=10, random_state=42)
df_sample_scaled = scaler_loaded.transform(df_sample)

# Start the batch processing
predictions =  kmeans_loaded.predict(df_sample_scaled)

# Store your predictions in the df_sample as a new column
df_sample['segment'] = predictions

## Task 4: Results Validation and Reporting
Create tools to validate your segmentation results and generate useful reports for the marketing team.

In [ ]:
# Create validation and reporting functions that:
# - Verify segment assignments
# - Generate segment summaries
# - Identify key characteristics per segment
def analyze_segmentation_results(df):
    """
    Analyzes segmentation results and generates reports
    Returns: Analysis summary
    """
    if 'segment' not in df.columns:
        raise ValueError("The DataFrame does not contain a 'segment' column.")

    # Total number of users per segment
    segment_counts = df['segment'].value_counts().sort_index()

    # Feature averages per segment
    segment_means = df.groupby('segment').mean(numeric_only=True)

    # Key characteristics per segment
    key_characteristics = {}
    for col in segment_means.columns:
        dominant_segment = segment_means[col].idxmax()
        key_characteristics[col] = f"Segment {dominant_segment} has the highest average {col}"

    # Print report
    print("\n=== Segmentation Analysis Report ===\n")

    print("Segment Counts:")
    for seg, count in segment_counts.items():
        print(f"  Segment {seg}: {count} users")

    print("\nFeature Averages by Segment:")
    for seg, row in segment_means.iterrows():
        print(f"  Segment {seg}:")
        for feature, value in row.items():
            print(f"    {feature}: {value:.2f}")

    print("\nKey Characteristics:")
    for feature, desc in key_characteristics.items():
        print(f"  - {desc}")

    return None

analyze_segmentation_results(df_sample)